<a href="https://colab.research.google.com/github/BassemRamdan/AI-Resume-Intelligence/blob/main/notebooks/07_Entity_Extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 7: Skill & Entity Extraction

**Goal:**
- Load the cleaned, lemmatized resume text produced in Phase 4 (`nlp_processed_resumes.csv`).
- Build an entity extraction pipeline that pulls out **Skills, Education, Experience, and Projects** from raw resume text.
- Follow the project rule: **do not hand-invent labeled training data.** Instead, rely on:
  1. **Zero-shot NER** via [GLiNER](https://github.com/urchade/GLiNER) — a pretrained model that accepts arbitrary entity labels at inference time, no fine-tuning required.
  2. A **curated skill taxonomy matcher** (spaCy `PhraseMatcher`) as a fast, high-precision complement for the "Skills" category.
  3. An optional **LLM-based extractor** (via the Anthropic API) for cases where GLiNER confidence is low or the resume is short/ambiguous.
- Output a structured, per-resume JSON/CSV of extracted entities that can seed bounding-box labels for LayoutLMv3 in a later phase.


In [1]:
!pip install gliner spacy pandas tqdm huggingface_hub anthropic -q
!python -m spacy download en_core_web_sm -q


[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: C:\Users\Delta\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')



[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: C:\Users\Delta\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## 1. Setup and Imports

In [2]:
import os
import re
import json
import getpass
import pandas as pd
import spacy
from tqdm.auto import tqdm
from gliner import GLiNER

pd.set_option("display.max_colwidth", 120)
print("Imports OK.")

C:\Users\Delta\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK.


## 2. Load Processed Resume Data from Phase 4

We use `raw_clean_text` (not the lemmatized version) as input to the entity extractors — lemmatization
strips case and word endings that NER models rely on (e.g. "Bachelor's", "Managed", proper nouns).

In [3]:
processed_path = r"D:\AI_Huawei_NTI\Final_project\AI-Resume-Intelligence\notebooks\nlp_processed_resumes.csv"

if not os.path.exists(processed_path):
    print("Please upload 'nlp_processed_resumes.csv' from Phase 4 to this Colab environment "
          "(or download it via hf_hub_download if you pushed it to the Hub).")
else:
    nlp_df = pd.read_csv(processed_path)
    nlp_df["raw_clean_text"] = nlp_df["raw_clean_text"].fillna("")
    print(f"Loaded {len(nlp_df)} processed resumes.")
    nlp_df.head(3)


nlp_df.head()

Loaded 2466 processed resumes.


,filename,category,raw_clean_text,lemmatized_text
0,10554236.pdf,ACCOUNTANT,"ACCOUNTANT Summary Financial Accountant specializing in financial planning, reporting and analysis within the Depart...",accountant summary financial accountant specialize financial planning reporting analysis department defense highligh...
1,10674770.pdf,ACCOUNTANT,STAFF ACCOUNTANT Summary Highly analytical and detail-oriented professional; possessing extensive financial statemen...,staff accountant summary highly analytical detail orient professional possess extensive financial statement backgrou...
2,11163645.pdf,ACCOUNTANT,"ACCOUNTANT Professional Summary To obtain a position in a fast-paced business office environment, demanding a strong...",accountant professional summary obtain position fast pace business office environment demand strong organizational t...
3,11759079.pdf,ACCOUNTANT,"SENIOR ACCOUNTANT Experience Company Name June 2011 to Current Senior Accountant City , State Prepare quarterly and ...",senior accountant experience company june current senior accountant city state prepare quarterly annual financial st...
4,12065211.pdf,ACCOUNTANT,SENIOR ACCOUNTANT Professional Summary Senior accountant who completes accounting activities with accuracy and speed...,senior accountant professional summary senior accountant complete accounting activity accuracy speed extensive exper...


## 3. Zero-Shot Entity Extraction with GLiNER

GLiNER lets us pass **any** label set at inference time — we are not training a classifier or
inventing ground-truth labels, we are asking a general-purpose model to find spans matching
open-vocabulary categories. This satisfies the "do not invent labels" constraint: the labels
below are just prompts describing what to look for, not a fixed taxonomy the model was fit to.

In [4]:
gliner_model = GLiNER.from_pretrained("urchade/gliner_multi-v2.1")

ENTITY_LABELS = [
    "skill",
    "degree",
    "university",
    "job title",
    "company",
    "project name",
    "certification",
    "programming language",
    "years of experience",
]

print("GLiNER loaded. Labels:", ENTITY_LABELS)

C:\Users\Delta\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\huggingface_hub\utils\_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 422.25it/s]


GLiNER loaded. Labels: ['skill', 'degree', 'university', 'job title', 'company', 'project name', 'certification', 'programming language', 'years of experience']


### Testing mode

Set `TEST_MODE = True` to run the pipeline on a small sample first (fast sanity check before
committing to the full dataset with GLiNER + LLM calls). Set it to `False` (or change `SAMPLE_SIZE`)
to run on everything.

In [5]:
# TEST_MODE = True
# SAMPLE_SIZE = 10

# if TEST_MODE:
#     nlp_df = nlp_df.sample(n=min(SAMPLE_SIZE, len(nlp_df)), random_state=42).reset_index(drop=True)
#     print(f"TEST_MODE on: running the pipeline on {len(nlp_df)} sampled resumes.")
# else:
#     print(f"TEST_MODE off: running the pipeline on the full {len(nlp_df)} resumes.")

### Chunking long resumes

GLiNER (like most transformer NER models) has a limited context window. Resume text reconstructed
from bounding boxes can run long, so we split on sentence-ish boundaries and extract per chunk,
then merge the results back together.

In [6]:
nlp_sentencizer = spacy.blank("en")
nlp_sentencizer.add_pipe("sentencizer")

def chunk_text(text, max_chars=1500):
    """Split text into chunks under max_chars, breaking on sentence boundaries where possible."""
    doc = nlp_sentencizer(text)
    chunks, current = [], ""
    for sent in doc.sents:
        s = sent.text.strip()
        if not s:
            continue
        if len(current) + len(s) + 1 > max_chars and current:
            chunks.append(current.strip())
            current = s
        else:
            current = f"{current} {s}".strip()
    if current:
        chunks.append(current.strip())
    return chunks if chunks else [text]


def extract_entities_gliner(text, labels=ENTITY_LABELS, threshold=0.4):
    """Run GLiNER over a (possibly long) resume text and return deduplicated entities."""
    all_entities = []
    for chunk in chunk_text(text):
        preds = gliner_model.predict_entities(chunk, labels, threshold=threshold)
        all_entities.extend(preds)

    # Deduplicate by (lowercased text, label)
    seen, deduped = set(), []
    for ent in all_entities:
        key = (ent["text"].strip().lower(), ent["label"])
        if key not in seen:
            seen.add(key)
            deduped.append({"text": ent["text"].strip(), "label": ent["label"], "score": round(ent["score"], 3)})
    return deduped

## 4. Run Extraction Pipeline Across All Resumes

In [7]:
extraction_results = []

for _, row in tqdm(nlp_df.iterrows(), total=len(nlp_df), desc="Zero-shot NER"):
    entities = extract_entities_gliner(row["raw_clean_text"])
    extraction_results.append({
        "filename": row["filename"],
        "category": row["category"],
        "entities": entities,
    })

print(f"Extracted entities for {len(extraction_results)} resumes.")
extraction_results[0]

Zero-shot NER:   1%|          | 22/2466 [03:03<5:10:49,  7.63s/it]C:\Users\Delta\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\gliner\data_processing\processor.py:422: UserWarning: Sentence of length 405 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
Zero-shot NER:   2%|▏         | 46/2466 [06:00<3:55:09,  5.83s/it]C:\Users\Delta\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\gliner\data_processing\processor.py:422: UserWarning: Sentence of length 491 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
Zero-shot NER:   4%|▍         | 102/2466 [11:33<3:39:21,  5.57s/it]C:\Users\Delta\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Pyth

Extracted entities for 2466 resumes.


{'filename': '10554236.pdf',
 'category': 'ACCOUNTANT',
 'entities': [{'text': 'Financial Accountant',
   'label': 'job title',
   'score': 0.672},
  {'text': 'Critical thinking', 'label': 'skill', 'score': 0.58},
  {'text': 'Accounting operations professional',
   'label': 'job title',
   'score': 0.495},
  {'text': 'DFAS Europe', 'label': 'company', 'score': 0.67},
  {'text': 'Company Name', 'label': 'company', 'score': 0.495},
  {'text': 'Accountant', 'label': 'job title', 'score': 0.425},
  {'text': 'Resource Advisor', 'label': 'job title', 'score': 0.711},
  {'text': 'Commander', 'label': 'job title', 'score': 0.564},
  {'text': 'Billing Official', 'label': 'job title', 'score': 0.726},
  {'text': 'Staff Accountant', 'label': 'job title', 'score': 0.794},
  {'text': 'HQ USAFE', 'label': 'company', 'score': 0.538},
  {'text': 'USAFE', 'label': 'university', 'score': 0.457},
  {'text': 'DFAS Limestone', 'label': 'company', 'score': 0.614},
  {'text': 'USAFE', 'label': 'company', 'sc

## 5. Structure Extracted Entities per Resume

Group the flat entity list into the four target buckets (Skills, Education, Experience, Projects)
so downstream consumers (and eventually LayoutLMv3 label generation) get a consistent schema.

In [14]:
LABEL_TO_BUCKET = {
    "skill": "skills",
    "programming language": "skills",
    "degree": "education",
    "university": "education",
    "job title": "experience",
    "company": "experience",
    "years of experience": "experience",
    "project name": "projects",
    "certification": "certifications",
}

def bucket_entities(entities):
    buckets = {"skills": [], "education": [], "experience": [], "projects": [], "certifications": []}
    for ent in entities:
        bucket = LABEL_TO_BUCKET.get(ent["label"])
        if bucket:
            buckets[bucket].append(ent["text"])
    # de-duplicate within each bucket, preserve order
    for k in buckets:
        buckets[k] = list(dict.fromkeys(buckets[k]))
    return buckets

structured_records = []
for rec in extraction_results:
    buckets = bucket_entities(rec["entities"])
    structured_records.append({
        "filename": rec["filename"],
        "category": rec["category"],
        **buckets,
    })

structured_df = pd.DataFrame(structured_records)
structured_df.head(5)

,filename,category,skills,education,experience,projects,certifications
0,10554236.pdf,ACCOUNTANT,"[Critical thinking, Managerial Accounting I, Auditing Methods and Concepts, Organizational Leadership, Management De...","[USAFE, GS-8, Northern Maine Community College, Husson College, Bachelors degree, Professional Military Comptroller ...","[Financial Accountant, Accounting operations professional, DFAS Europe, Company Name, Accountant, Resource Advisor, ...",[],"[Certified Defense Financial Manager, CDFM]"
1,10674770.pdf,ACCOUNTANT,[],"[Bachelor of Science, University of North Carolina]","[STAFF ACCOUNTANT, DBA, Company Name, Cary Keisler Inc., American Express, VP of Finance, Financial Management Partn...",[project],[]
2,11163645.pdf,ACCOUNTANT,"[analytical aptitude, Access, Excel, Outlook, PowerPoint, Word]",[],"[ACCOUNTANT, Company Name, Accounts Receivable Clerk, Company Name ï¼, Mortgage Underwriter, Commercial Auto Underwr...",[],[]
3,11759079.pdf,ACCOUNTANT,"[Microsoft Excel, programming, writing skills]","[EMORY UNIVERSITY, Goizueta Business School, Bachelor of Business Administration, Accounting]","[SENIOR ACCOUNTANT, Company Name, Associate Fund Controller, MSREF, Morgan Stanley Real Estate Funds 6I, Information...",[],"[CFE, Certified Fraud Examiner]"
4,12065211.pdf,ACCOUNTANT,[],"[Bachelor of Business Administration, TEMPLE UNIVERSITY]","[SENIOR ACCOUNTANT, Deloitte, Accountant, Company Name, Financial Analyst, Palm Desert National Bank, Innobeta Syste...",[],[]


## 6. Complementary Skill Matching (Curated Taxonomy)

Zero-shot NER can miss short, jargon-heavy skill tokens (e.g. "AWS", "React", "Pandas"). As a
precision boost — not a replacement — we match against a curated skill vocabulary with spaCy's
`PhraseMatcher`. This is a lookup, not a trained/invented label set: swap in any external skills
taxonomy (e.g. ESCO, LinkedIn Skills, O*NET) in place of `SKILL_VOCAB` for production use.

In [15]:
# Placeholder starter vocabulary — replace with a full taxonomy (ESCO / O*NET / LinkedIn Skills) for production.
SKILL_VOCAB = [
    "Python", "Java", "SQL", "AWS", "Azure", "GCP", "Docker", "Kubernetes",
    "React", "Node.js", "TensorFlow", "PyTorch", "Pandas", "NumPy", "spaCy",
    "Excel", "PowerPoint", "Tableau", "Power BI", "Git", "Linux", "C++",
    "Machine Learning", "Deep Learning", "NLP", "Data Analysis", "Project Management",
]

matcher_nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])
from spacy.matcher import PhraseMatcher

matcher = PhraseMatcher(matcher_nlp.vocab, attr="LOWER")
matcher.add("SKILL", [matcher_nlp.make_doc(s) for s in SKILL_VOCAB])

def match_curated_skills(text):
    doc = matcher_nlp.make_doc(text)
    matches = matcher(doc)
    found = {doc[start:end].text for _, start, end in matches}
    return sorted(found)

# Merge curated matches into the GLiNER skill lists
for i, row in structured_df.iterrows():
    resume_text = nlp_df.loc[nlp_df["filename"] == row["filename"], "raw_clean_text"].values[0]
    curated = match_curated_skills(resume_text)
    merged = list(dict.fromkeys(row["skills"] + curated))
    structured_df.at[i, "skills"] = merged

structured_df[["filename", "skills"]].head(5)

,filename,skills
0,10554236.pdf,"[Critical thinking, Managerial Accounting I, Auditing Methods and Concepts, Organizational Leadership, Management De..."
1,10674770.pdf,"[Excel, excel]"
2,11163645.pdf,"[analytical aptitude, Access, Excel, Outlook, PowerPoint, Word]"
3,11759079.pdf,"[Microsoft Excel, programming, writing skills, Excel]"
4,12065211.pdf,"[Excel, SQL]"


## 8. Save Final Structured Dataset

In [16]:
structured_df.to_csv("resume_entities.csv", index=False)

with open("resume_entities.json", "w", encoding="utf-8") as f:
    json.dump(structured_df.to_dict(orient="records"), f, indent=2, ensure_ascii=False)

print(f"Saved 'resume_entities.csv' and 'resume_entities.json'. Shape: {structured_df.shape}")

# Optional: push to Hugging Face Hub
# from huggingface_hub import HfApi
# HfApi().upload_file(path_or_fileobj="resume_entities.json", path_in_repo="resume_entities.json",
#                      repo_id="<your-username>/ai-resume-intelligence", repo_type="dataset")

Saved 'resume_entities.csv' and 'resume_entities.json'. Shape: (2466, 7)


## 9. Quick Analysis

In [ ]:
from collections import Counter

all_skills = [s for skills in structured_df["skills"] for s in skills]
top_skills = Counter(all_skills).most_common(20)

print("Top 20 extracted skills across all resumes:")
for skill, count in top_skills:
    print(f"{skill}: {count}")

print("\nEntities per resume (avg):")
for bucket in ["skills", "education", "experience", "projects", "certifications"]:
    avg = structured_df[bucket].apply(len).mean()
    print(f"{bucket}: {avg:.2f} avg entities/resume")

print("\nNext step (Phase 8): use these entity spans, combined with the bounding-box coordinates "
      "from Phase 3, to auto-generate LayoutLMv3 token labels.")

Top 20 extracted skills across all resumes:
Excel: 892
PowerPoint: 430
Project Management: 311
project management: 259
Skills: 192
SQL: 177
Project management: 152
Microsoft Office: 128
Word: 122
Access: 111
excel: 110
C: 108
HTML: 106
English: 97
Microsoft Word: 89
customer service: 87
Excellent communication skills: 86
quality: 85
Leadership: 84
leadership: 83

Entities per resume (avg):
skills: 7.68 avg entities/resume
education: 3.69 avg entities/resume
experience: 11.46 avg entities/resume
projects: 0.26 avg entities/resume
certifications: 0.75 avg entities/resume

Next step (Phase 8): use these entity spans, combined with the bounding-box coordinates from Phase 3, to auto-generate LayoutLMv3 token labels.


: 